## Project 1: Building a Recommender System for Movie Ratings

**Course:** DATA 612 – Recommender Systems<br>
**Student:** Inna Yedzinovich  

For this project, I am building a simple recommender system that predicts user ratings for movies, similar to a Rotten Tomatoes-style rating system. The goal is to estimate how users would rate movies based on available data, even when some ratings are missing. This will help simulate how real platforms generate recommendations.

For this purpose, I created a small dataset based on movie reviews inspired by Rotten Tomatoes. The dataset includes 5 users and several movies such as Backrooms, Obsession, Exit 8, The Drama, and Silent Friend. I simplified the critic and audience feedback into numeric ratings (1–5 scale) and included some missing values to reflect real-world situations where not all users rate every movie.

The following matrix represents user ratings for movies on a 1–5 scale. Each row represents a user, and each column represents a movie. Missing values indicate that the user has not rated that movie.

In [1]:
# imports
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
# synthetic user-item rating matrix
data = pd.DataFrame({
    'Backrooms':     [4, 5, np.nan, 3, 2],
    'Obsession':     [2, np.nan, 4, 3, 3],
    'Exit 8':        [5, 4, 4, np.nan, 3],
    'The Drama':     [3, 2, np.nan, 4, 2],
    'Silent Friend': [4, np.nan, 5, 3, 4]
}, index = [
    "Ty Burr",
    "Coleman Spilde",
    "Lauren Veneziani",
    "Justin Chang",
    "Grace Randolph"
])

data

,Backrooms,Obsession,Exit 8,The Drama,Silent Friend
Ty Burr,4.0,2.0,5.0,3.0,4.0
Coleman Spilde,5.0,NaN,4.0,2.0,NaN
Lauren Veneziani,NaN,4.0,4.0,NaN,5.0
Justin Chang,3.0,3.0,NaN,4.0,3.0
Grace Randolph,2.0,3.0,3.0,2.0,4.0


The dataset is split into training and test sets by removing a few known ratings and placing them into the test set. The training set is used to build the model, while the test set is used to evaluate how well the model predicts unseen data.

In [3]:
train = data.copy()
test = pd.DataFrame(np.nan, index=data.index, columns=data.columns)

# move some values to test set
test.loc['Ty Burr', 'Exit 8'] = train.loc['Ty Burr', 'Exit 8']
train.loc['Ty Burr', 'Exit 8'] = np.nan

test.loc['Lauren Veneziani', 'Silent Friend'] = train.loc['Lauren Veneziani', 'Silent Friend']
train.loc['Lauren Veneziani', 'Silent Friend'] = np.nan

test.loc['Grace Randolph', 'Backrooms'] = train.loc['Grace Randolph', 'Backrooms']
train.loc['Grace Randolph', 'Backrooms'] = np.nan


print("Training Set:")
print(train)

print("\nTest Set:")
print(test)


Training Set:
                  Backrooms  Obsession  Exit 8  The Drama  Silent Friend
Ty Burr                 4.0        2.0     NaN        3.0            4.0
Coleman Spilde          5.0        NaN     4.0        2.0            NaN
Lauren Veneziani        NaN        4.0     4.0        NaN            NaN
Justin Chang            3.0        3.0     NaN        4.0            3.0
Grace Randolph          NaN        3.0     3.0        2.0            4.0

Test Set:
                  Backrooms  Obsession  Exit 8  The Drama  Silent Friend
Ty Burr                 NaN        NaN     5.0        NaN            NaN
Coleman Spilde          NaN        NaN     NaN        NaN            NaN
Lauren Veneziani        NaN        NaN     NaN        NaN            5.0
Justin Chang            NaN        NaN     NaN        NaN            NaN
Grace Randolph          2.0        NaN     NaN        NaN            NaN


Now I want to ignore users and movies and just predict the same average for everything as it is shown in the video.



In [4]:
global_mean = train.stack().mean()
print("Global Mean:", global_mean)

Global Mean: 3.3529411764705883


After that I try predict ALL missing values using this mean.

In [5]:
predictions = train.copy()
predictions = predictions.fillna(global_mean)

print("Predictions (Average Model):")
print(predictions)


Predictions (Average Model):
                  Backrooms  Obsession    Exit 8  The Drama  Silent Friend
Ty Burr            4.000000   2.000000  3.352941   3.000000       4.000000
Coleman Spilde     5.000000   3.352941  4.000000   2.000000       3.352941
Lauren Veneziani   3.352941   4.000000  4.000000   3.352941       3.352941
Justin Chang       3.000000   3.000000  3.352941   4.000000       3.000000
Grace Randolph     3.352941   3.000000  3.000000   2.000000       4.000000


At this point, I want to measure how far your predictions are from actual values. Formula: RMSE = sqrt(mean((actual - predicted)^2)). 

RMSE is calculated by using the global average as the prediction for all missing ratings. This means we do not use the real values directly, and we can properly measure how far our predictions are from the actual ratings.

In [6]:
user_bias = train.mean(axis=1) - global_mean

print("User Bias:")
print(user_bias)

User Bias:
Ty Burr            -0.102941
Coleman Spilde      0.313725
Lauren Veneziani    0.647059
Justin Chang       -0.102941
Grace Randolph     -0.352941
dtype: float64


In [7]:
# train RMSE
train_errors = ((train - global_mean) ** 2).stack()
rmse_train = np.sqrt(train_errors.mean())

print("RMSE (Train):", rmse_train)


# test RMSE
test_errors = ((test - global_mean) ** 2).stack()
rmse_test = np.sqrt(test_errors.mean())

print("RMSE (Test):", rmse_test)

RMSE (Train): 0.8360394355030527
RMSE (Test): 1.5552122431061512


Compute User Bias and Item Bias: Instead of predicting everything as the same average, I want to adjust by the following formula: prediction = global_mean + user_bias + item_bias. 

User bias measures how much each user tends to rate above or below the global average. Item bias measures how much each movie is rated above or below the global average. These biases help improve predictions by adjusting for differences in user behavior and item popularity.

In [8]:
user_bias = train.mean(axis=1) - global_mean

print("User Bias:")
print(user_bias)


item_bias = train.mean(axis=0) - global_mean

print("\nItem Bias:")
print(item_bias)


User Bias:
Ty Burr            -0.102941
Coleman Spilde      0.313725
Lauren Veneziani    0.647059
Justin Chang       -0.102941
Grace Randolph     -0.352941
dtype: float64

Item Bias:
Backrooms        0.647059
Obsession       -0.352941
Exit 8           0.313725
The Drama       -0.602941
Silent Friend    0.313725
dtype: float64


Build Baseline Predictions: prediction = global_mean + user_bias + item_bias. 

The baseline predictor improves the model by adjusting the global average with user bias and item bias. This allows predictions to account for both individual user behavior and overall movie popularity, making the recommendations more accurate.

In [9]:
# create empty prediction matrix
baseline_pred = pd.DataFrame(index=train.index, columns=train.columns)

# fill predictions matrix
for user in train.index:
    for item in train.columns:
        baseline_pred.loc[user, item] = (
            global_mean + user_bias[user] + item_bias[item]
        )

baseline_pred = baseline_pred.astype(float)

print("Baseline Predictions:")
print(baseline_pred)

Baseline Predictions:
                  Backrooms  Obsession    Exit 8  The Drama  Silent Friend
Ty Burr            3.897059   2.897059  3.563725   2.647059       3.563725
Coleman Spilde     4.313725   3.313725  3.980392   3.063725       3.980392
Lauren Veneziani   4.647059   3.647059  4.313725   3.397059       4.313725
Justin Chang       3.897059   2.897059  3.563725   2.647059       3.563725
Grace Randolph     3.647059   2.647059  3.313725   2.397059       3.313725


Now, I want to compute RMSE again to evaluate it the same way as before to see how RME has changed.

In [10]:
# train RMSE
train_errors = ((train - baseline_pred) ** 2).stack()
rmse_train_baseline = np.sqrt(train_errors.mean())

print("Baseline RMSE (Train):", rmse_train_baseline)


# test RMSE
test_errors = ((test - baseline_pred) ** 2).stack()
rmse_test_baseline = np.sqrt(test_errors.mean())

print("Baseline RMSE (Test):", rmse_test_baseline)

Baseline RMSE (Train): 0.6311167427227875
Baseline RMSE (Test): 1.3224547803844684


I want to explain what is happening here. First, I used the global average to predict all missing ratings and calculated RMSE. The error was relatively higher because the model treats all users and movies the same, without considering any differences. Then I improved the model by adding user bias and item bias, which adjust the predictions based on how each user rates and how each movie is generally rated. 

After including these adjustments, both the train and test RMSE decreased. This shows that the model became more accurate because it is no longer using a single average, but instead makes more personalized predictions.


## Summary

In this project, I built a simple recommender system to predict movie ratings using a user-item matrix. First, I used the global average rating to make predictions and calculated the RMSE. This method was simple but not very accurate.

Then, I improved the model by adding user bias and item bias. This helped adjust the predictions based on how users rate and how movies are generally rated. After doing this, the RMSE became lower, which means the predictions improved.

Overall, the baseline model worked better than using just the average. This shows that including user and item differences makes the recommendations more accurate.
